# Build ChromaDB trên Colab GPU (baseline RAG hỏi đáp thuốc)

Đầu vào: `storage/<tên file csv>/drugs.sqlite` đã có sẵn ở máy (bảng `drugs` + `chunks` do `build_index.py` tạo).
Đầu ra: `chroma.zip` để giải nén vào `projects/base-rag/storage/<tên file csv>/` trên máy.

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**. Sau đó Runtime → Run all.

Các tham số phải khớp với code local (`rag/config.py`, `rag/text_embedder.py`, `rag/chroma_vector_store.py`):
- chromadb **1.5.9** (cùng bản với `.venv` local)
- model `intfloat/multilingual-e5-small`, tiền tố `passage: `, vector chuẩn hoá
- collection `drug_chunks`, khoảng cách cosine, metadata `ndc` + `section`

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi -L

In [ ]:
# 2. Cài thư viện (chromadb phải cùng phiên bản với máy local để đọc được thư mục chroma)
!pip install -q "chromadb==1.5.9" "sentence-transformers>=3.0"

In [ ]:
# 3. Cấu hình
USE_DRIVE = False  # True: đọc sqlite / ghi chroma.zip qua Google Drive (nên dùng nếu upload/download chậm)
DRIVE_DIR = "/content/drive/MyDrive/base-rag"  # thư mục trên Drive chứa drugs.sqlite khi USE_DRIVE=True

SQLITE_PATH = f"{DRIVE_DIR}/drugs.sqlite" if USE_DRIVE else "/content/drugs.sqlite"
OUTPUT_DIR = "/content/storage"
CHROMA_DIR = f"{OUTPUT_DIR}/chroma"
CHROMA_COLLECTION = "drug_chunks"  # khớp rag/config.py
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"  # khớp EMBEDDING_MODEL local
BATCH_SIZE = 256  # batch embed trên GPU
ADD_BATCH = 4000  # batch ghi vào Chroma (giới hạn của Chroma ~5461)
USE_FP16 = True  # nhanh hơn ~2x trên T4, sai số không đáng kể với vector đã chuẩn hoá

In [ ]:
# 4. Lấy file drugs.sqlite (Drive hoặc upload trực tiếp)
import os
import shutil

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if not os.path.exists(SQLITE_PATH):
    from google.colab import files
    print("Chọn file storage/drugs.sqlite trên máy để upload...")
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    if os.path.abspath(uploaded_name) != SQLITE_PATH:
        shutil.move(uploaded_name, SQLITE_PATH)

print(f"SQLite: {SQLITE_PATH} ({os.path.getsize(SQLITE_PATH) / 1e6:.1f} MB)")

In [ ]:
# 5. Đọc toàn bộ chunk từ SQLite (sắp theo chunk_id cho ổn định)
import sqlite3

conn = sqlite3.connect(SQLITE_PATH)
rows = conn.execute("SELECT chunk_id, ndc, section, text FROM chunks ORDER BY chunk_id").fetchall()
n_drugs = conn.execute("SELECT COUNT(*) FROM drugs").fetchone()[0]
conn.close()
print(f"{n_drugs} thuốc, {len(rows)} chunk")

In [ ]:
# 6. Embedding trên GPU (giống rag/text_embedder.py: tiền tố 'passage: ' + normalize)
import time
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
model = SentenceTransformer(EMBEDDING_MODEL, device=device)
if USE_FP16 and device == "cuda":
    model.half()

started = time.time()
embeddings = model.encode(
    [f"passage: {text}" for _, _, _, text in rows],
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype("float32")
print(f"Embed xong {embeddings.shape} trong {time.time() - started:.0f}s")

In [ ]:
# 7. Ghi vào ChromaDB mới (giống rag/chroma_vector_store.py)
import chromadb
from tqdm.auto import tqdm

shutil.rmtree(CHROMA_DIR, ignore_errors=True)
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_or_create_collection(
    name=CHROMA_COLLECTION,
    embedding_function=None,
    metadata={"hnsw:space": "cosine"},
)

for start in tqdm(range(0, len(rows), ADD_BATCH), desc="Ghi Chroma"):
    batch = rows[start : start + ADD_BATCH]
    collection.add(
        ids=[chunk_id for chunk_id, _, _, _ in batch],
        embeddings=embeddings[start : start + ADD_BATCH].tolist(),
        metadatas=[{"ndc": ndc, "section": section} for _, ndc, section, _ in batch],
    )

print("Vector trong Chroma:", collection.count())
assert collection.count() == len(rows), "Số vector không khớp số chunk"

In [ ]:
# 8. Thử truy xuất 1 câu hỏi tiếng Việt (không gọi LLM)
question = "Metformin chống chỉ định với những trường hợp nào?"
query_vec = model.encode([f"query: {question}"], normalize_embeddings=True).astype("float32").tolist()
result = collection.query(query_embeddings=query_vec, n_results=5, include=["distances"])

conn = sqlite3.connect(SQLITE_PATH)
for chunk_id, distance in zip(result["ids"][0], result["distances"][0]):
    name = conn.execute(
        "SELECT d.ten_thuoc FROM chunks c JOIN drugs d ON d.ndc = c.ndc WHERE c.chunk_id = ?",
        (chunk_id,),
    ).fetchone()[0]
    print(f"{1 - distance:.3f}  {name:40.40}  {chunk_id}")
conn.close()

In [ ]:
# 9. Nén thư mục chroma -> chroma.zip (bên trong là thư mục chroma/) rồi tải về
del collection, client
zip_path = shutil.make_archive("/content/chroma", "zip", root_dir=OUTPUT_DIR, base_dir="chroma")
print(f"{zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

if USE_DRIVE:
    shutil.copy(zip_path, f"{DRIVE_DIR}/chroma.zip")
    print(f"Đã chép sang Drive: {DRIVE_DIR}/chroma.zip")
else:
    from google.colab import files
    files.download(zip_path)

## Trên máy local sau khi tải `chroma.zip`

1. Đảm bảo không còn `build_index.py` hay `streamlit` đang chạy.
2. Xoá thư mục cũ `projects/base-rag/storage/<tên file csv>/chroma` (giữ nguyên `drugs.sqlite`).
3. Giải nén `chroma.zip` vào `projects/base-rag/storage/<tên file csv>/` → được `.../chroma/...`.
4. `streamlit run app.py` — sidebar sẽ hiện số vector bằng số chunk ở bước 5.

Lưu ý: KHÔNG chạy lại `build_index.py --reset` sau đó (sẽ xoá chroma vừa tải). Nếu build lại SQLite thì phải chạy lại notebook này.